# Testing out the redaction capabilities


In [12]:
import os
import sys
import pandas as pd

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../'))
if root_path not in sys.path:
    sys.path.insert(0, root_path)
for line in sys.path: 
    print(line)

c:\Users\danie\Code\aiticketgrouper
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\python313.zip
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\DLLs
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64\Lib
C:\Users\danie\AppData\Local\Python\pythoncore-3.13-64
c:\Users\danie\Code\aiticketgrouper\.venv

c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\win32
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\win32\lib
c:\Users\danie\Code\aiticketgrouper\.venv\Lib\site-packages\Pythonwin


In [18]:
csv_path = "../../external_files/incident_ticket_data.csv" # go up two levels to get external files

df = pd.read_csv(csv_path)

if "Description" not in df.columns:
    raise ValueError("CSV must contain 'Description' column")

df = df[:1000]

df.head()

,Number,Description,Opened by,Company,ITSM Department,Created,Urgency,Impact,Priority,Assignment group,...,User input,Comments and Work notes,Manday Effort (hrs),Ticket Type,AMS Domain,AMS System Type,AMS Category Type,AMS Service Type,AMS Business Related,AMS IT Related
0,INC0010171,GRPT not working as expected. ZMMM_PO_REV is n...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 9:07,2 - Medium,3 - Low,4 - Low,PISCAP L2 SD BRS,...,NaN,2025-04-11 13:26:58 - BALAKUMAR GANESAN (Addit...,3.0,Issue,IS,S4HANA,Non-Genesis,Business-Related,BZ-B12-Master Data (Wrong Maintenance),NaN
1,INC0010181,eTR-S1-24000073\r\nExchange Rate did not auto ...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 9:51,2 - Medium,3 - Low,4 - Low,PISCAP L2 Workflow (SN),...,NaN,2024-04-05 02:49:36 - Reeman Mathur (Additiona...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
2,INC0010188,There is no GRPT maintenance for Sold-To: 3901...,Indah Humairah Sulaiman,PIDSAP,PIDSAP,18/3/24 10:19,3 - Low,3 - Low,4 - Low,PISCAP L2 SD BRS,...,NaN,2024-05-13 12:57:15 - BALAKUMAR GANESAN (Addit...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
3,INC0010189,Interface\t fpl\r\nSubsidiary\t...,Chenxing Cao,PA,PISCAP,18/3/24 10:24,3 - Low,3 - Low,4 - Low,PISCAP L2 Mulesoft/SOA,...,NaN,2024-03-18 10:30:07 - Chenxing Cao (Work notes...,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN
4,INC0010192,"retrieve new SAP password, thank you.",SOOK FONG NG,PM,PM,18/3/24 10:33,1 - High,3 - Low,3 - Moderate,PISCAP L2 SAP BASIS,...,NaN,NaN,NaN,Issue,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.shape

(1000, 30)

!python -m spacy download en_core_web_sm

## CleaningService Class
Cleans the text by removing stop words, phrases, expanding acronyms, and other text normalization techniques.

In [20]:
import re

class CleaningService:
    # words to remove
    stop_words = {"hi", "hello", "dear", "hey", "regards", "please", "pls", "thank", 
                        "thanks", "kindly", "kind", "bye"}
    
    stop_phrases = {"thank you", "good day", "dear team", "pls kindly advise", "(Work notes)", "(Additional comments)"}
    
    # words to replace 
    acronym_expansions = {
        "asap": "as soon as possible",
        "cr": "change request",
        "vs": "visual studio",
        "pc": "computer",
        "com": "computer",
        "â€“": "-" # unicode dash fix
        }
    
    case_sensitive_acronyms = {
        #"PO": "purchase order", 
        "SO": "sales order",  
        "DO": "delivery order",
        "IT": "information technology",
        "PAS": "project accounting system",
        "PIR": "purchase information record"
        }
    
    # get rid of these placeholders when i figure out how to load from config file
    custom_acronym_placeholder = {
        "purchase order": ["po", "p.o.", "p o", "p/o"],
        "salesforce": ["sfdc"],
        }
    
    def __init__(self, custom_stop_words: set[str] | None = None, custom_acronyms: dict[str, list[str]] | None = None) -> None:
        if custom_stop_words:
            self.stop_words.update(custom_stop_words) # adds custom ones from the settings

        self.pattern = None
        if custom_acronyms:
            self._add_custom_acronyms(custom_acronyms)
        else:
            self._add_custom_acronyms(self.custom_acronym_placeholder)
    
    
    def _add_custom_acronyms(self, custom_acronyms: dict[str, list[str]]) -> None:
        """Adds an inverted many-to-one mapping of custom acronyms to the 'pattern' class attribute.  
        Many-to-one creates multiple acronym-expansion pairs to speed up queries, reducing reference timings to O(1).
        
        
        For example, "purchase order" that has expansions: ["po", "p.o.", "p o"] 
        becomes three separate entries, {"po": "purchase order", "p.o.": "purchase order", "p o": "purchase order"}
        Args:
            custom_acronyms (dict[str, List[str]]): Dict mapping of expansions to their matching colloquialism/shorthand/jargon.  
                Key: expansion word, List values: target words  
                *Example*: {"purchase order": ["po", "p.o.", "p o", "p/o"]}
        """
        custom_flipped_dict = {}
        # inversion logic
        for expansion, acronyms in custom_acronyms.items():
            for acr in acronyms:
                custom_flipped_dict[acr.lower()] = expansion
        
        self.acronym_expansions.update(custom_flipped_dict) # merge with standard acronyms
        escaped_keys: list[str] = [re.escape(k) for k in self.acronym_expansions.keys()] # escape the custom keys for regex pattern matching 
        regex_pattern = fr"\b({'|'.join(escaped_keys)})\b" # word boundary to avoid partial matches like "com" in "communication"
        self.pattern = re.compile(regex_pattern, flags=re.IGNORECASE)

    
    def remove_stop_words(self, text: str) -> str:
        """Removes stop words from input text to reduce artificial inflation of clustering similarity.
        
        Example: "Hi, please process the purchase order of 5000 bytes." - "Hi", "please" -> "process the purchase order of 5000 bytes."
        """
        if self.stop_phrases:
            phrase_pattern = r'\b(' + '|'.join([re.escape(p) for p in self.stop_phrases]) + r')\b'
            text = re.sub(phrase_pattern, '', text, flags=re.IGNORECASE)

        # remove standard pattern text
        text = re.sub(r'\b\d+\s*bytes\b', '', text)
        
        words = text.split()
        cleaned_words = [w for w in words if w not in self.stop_words]
        
        return " ".join(cleaned_words)
    
    
    def expand_acronyms(self, text:str) -> str:
        """Replaces short-hand words with full forms to aid with embedding quality.
        
        Example: "po" -> "purchase order"
        """
        if not self.pattern or not text:
            return text
        
        return self.pattern.sub(
            lambda match: self.acronym_expansions[match.group(0).lower()], 
            text
            )
    
        
    def expand_case_sensitive_acronyms(self, text: str) -> str:
        """Expands only case-sensitive acronyms to avoid false positives.
        
        Example: "DO" -> "delivery order", but "do" remains "do"."""
        for acr, expansion in self.case_sensitive_acronyms.items():
            text = re.sub(r'\b' + re.escape(acr) + r'\b', expansion, text)
        return text
        
    # TODO add custom PII patterns to catch the ITSM ticket numbers 
    def custom_pii_patterns(self, text: str) -> str:
        purchase_order_pattern = r'(?i)\bp[\/.]?o[\s#:]*\d{8,}\b'
        sales_order_pattern = r'(?i)\bs[\/.]?o[\s#:]*\d{9,}\b'
        invoice_pattern = r'(?i)invoice\s*#[\s:]*\d+'
        delivery_order_pattern = r'(?i)delivery\s*order#?[\s:]*\d+'
        external_invoice_pattern = r'(?i)external\s*inv(?:oice)?\s*num[\s:]*\d+'
        
        patterns = {
            purchase_order_pattern: "<PURCHASE_ORDER_ID>",
            sales_order_pattern: "<SALES_ORDER_ID>",
            invoice_pattern: "<INVOICE_NUMBER>",
            delivery_order_pattern: "<DELIVERY_ORDER>",
            external_invoice_pattern: "<EXTERNAL_INVOICE_NUMBER>"
        }

        for pattern, replacement in patterns.items():
            text = re.sub(pattern, replacement, text)
        return text
        
    
    def clean_text(self, text: str) -> str:
        """Cleans the input text by removing stop words and replacing acronyms."""
        if not text:
            raise ValueError("Input text cannot be empty.")
        
        text = self.expand_case_sensitive_acronyms(text)
        text = self.custom_pii_patterns(text)
        text = text.lower() # after case sensitive expansion, lower everything for uniformity
        text = self.remove_stop_words(text)
        text = self.expand_acronyms(text)
        
        return text


In [21]:
import spacy
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


nlp = spacy.load("en_core_web_sm")

class RedactionService:
    def __init__(self):
        self.analyzer = AnalyzerEngine()
        self.anonymizer = AnonymizerEngine()

        self.allow_list = ["eWorkplace"]
        
        custom_patterns = [
            Pattern(
                name="servicenow_inc_ticket",
                regex=r"\b(?i)(INC|RITM|CHG|TASK|PRB|WO)\d{7}\b",  # Example: INC1234567
                score=0.8
            )
        ]
        
        ticket_recognizer = PatternRecognizer(
            supported_entity="ITSM_TICKET",
            patterns=custom_patterns,
        )

        self.analyzer.registry.add_recognizer(ticket_recognizer)
        # self.analyzer.registry.add_recognizer(purchase_order_recognizer)

    def redact_text(self, text: str) -> str:
        # Analyze the text to find PII entities
        results = self.analyzer.analyze(
            text=text, 
            entities=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "LOCATION", "ITSM_TICKET"],
            language='en',
            allow_list=self.allow_list
            )

        # Configure custom anonymization to replace detected entities with placeholders, 
        operators={
            # replaces all other PII entities with default placeholders: "<PERSON>", "<EMAIL_ADDRESS>", etc.
            "ITSM_TICKET": OperatorConfig("replace", {"new_value": "<TICKET_ID>"}),  # Custom replacement for ITSM tickets
            "PURCHASE_ORDER": OperatorConfig("replace", {"new_value": "<PURCHASE_ORDER_ID>"}),
        }
        
        # Anonymize the text
        redacted_text = self.anonymizer.anonymize(
            text=text, 
            analyzer_results=results,  # type: ignore BULLSHIT WARNING BRUH FYM List[RecognizerResult] cannot case to List[RecognizerResult]
            operators=operators).text

        return redacted_text
    
        
    def semantic_expansion(self, text: str) -> str:
        """Expands grammar-specific abbreviations and acronyms while avoiding normal words even when input text is lower case.
        
        Example:
            Not exapanded: "Please *do* (verb) the task." -> "do" 
            Expanded: "the *DO*(noun) is missing" -> "delivery order"
            
        **Warning**: this method has performance implications due to the NLP processing involved. 
        For faster processing with potential accuracy tradeoffs, use case-sensitive recognition: "DO"-> "delivery order".
        """
        noun_propn_patterns = {
            "do": "delivery order",
            "so": "sales order",
        }
        
        doc = nlp(text)
        new_words = []
        
        for token in doc:
            if token.text.lower() in noun_propn_patterns and token.pos_ in {"NOUN", "PROPN"}:
                new_words.append(noun_propn_patterns[token.text.lower()])
            else:
                new_words.append(token.text_with_ws)
        
        
        return " ".join(new_words)
        
    

In [22]:
redact = RedactionService()
cleaner = CleaningService()

print(f"Cleaning Pattern: {cleaner.pattern}")

redacted_data_list = []
cleaned_data_list = []
df = df[['Number', 'Description', 'Created', 'Service', 'Service offering', 'Category', 'Subcategory', 'Assignment group']]


Cleaning Pattern: re.compile('\\b(asap|cr|vs|pc|com|â€“|po|p\\.o\\.|p\\ o|p/o|sfdc)\\b', re.IGNORECASE)


In [23]:
column_names = {
    "id": "Number",
    "description": "Description",
    "include_in_desc": ["Service", "Service offering", "Category", "Subcategory", "Assignment group"],
    "created_at": "Created"
}

config = {
    "collection_name": "snow_tickets",
    "insert_count": 1000,
    "redaction_required": True
}

extra_cols = column_names['include_in_desc']
valid_cols = [col for col in extra_cols if col in df.columns] # only include cols that exist in the df

if valid_cols: 
    extra_info_series = (df[valid_cols].astype(str).agg(' | '.join, axis=1).fillna(""))
    df[column_names["description"]] = df[column_names["description"]] + " [Context: " + extra_info_series + "]"


df['cleaned_text'] = df[column_names["description"]].apply(lambda x: cleaner.clean_text(x))
df['cleaned_text'] = df['cleaned_text'].apply(lambda x: redact.redact_text(x))

df.head()

,Number,Description,Created,Service,Service offering,Category,Subcategory,Assignment group,cleaned_text
0,INC0010171,GRPT not working as expected. ZMMM_PO_REV is n...,18/3/24 9:07,SAP,MM - Material Management,Application/Software,Data - Internal/External,PISCAP L2 SD BRS,grpt not working as expected. zmmm_po_rev is n...
1,INC0010181,eTR-S1-24000073\r\nExchange Rate did not auto ...,18/3/24 9:51,eWorkplace (eWorkplace SharePoint & eWorkplace...,NaN,Application/Software,Configuration,PISCAP L2 Workflow (SN),etr-s1-24000073 exchange rate did not auto pop...
2,INC0010188,There is no GRPT maintenance for Sold-To: 3901...,18/3/24 10:19,SAP,MM - Material Management,Application/Software,Data - Internal/External,PISCAP L2 SD BRS,there is no grpt maintenance for sold-to: 3901...
3,INC0010189,Interface\t fpl\r\nSubsidiary\t...,18/3/24 10:24,Mulesoft/EAI,NaN,Network,Configuration,PISCAP L2 Mulesoft/SOA,interface fpl subsidiary pidsap api name pana-...
4,INC0010192,"retrieve new SAP password, thank you. [Context...",18/3/24 10:33,SAP,BC - Basis,Application/Software,NaN,PISCAP L2 SAP BASIS,"retrieve new sap password, . [context: sap | b..."


Redaction Script

In [24]:
output_path = "../../external_files/incident_ticket_data_redacted.csv"
df.to_csv(output_path, index=False)
print(f"DataFrame saved to {output_path}")

DataFrame saved to ../../external_files/incident_ticket_data_redacted.csv
